# Process cleaning data for Predictive Dynamic Staffing for Food Delivery Operations

## Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os



## Load dataset

In [ ]:
df = pd.read_csv('food_delivery_orders_dataset.csv')

Path to dataset files: C:\Users\ASUS\.cache\kagglehub\datasets\razanihababdellatif\food-delivery-orders-and-eta-logistics-dataset\versions\1
Using dataset file: C:\Users\ASUS\.cache\kagglehub\datasets\razanihababdellatif\food-delivery-orders-and-eta-logistics-dataset\versions\1\food_delivery_orders_dataset.csv


In [6]:
df.head()

NameError: name 'df' is not defined

In [ ]:
# 1. Filter restoran sibuk (opsional, sesuaikan dengan kepadatan datamu)
order_counts = df['restaurant_id'].value_counts()
busy_restaurants = order_counts[order_counts > 4000].index
df_filtered = df[df['restaurant_id'].isin(busy_restaurants)].copy()

# 2. Agregasi target antrian dan fitur ML dalam window 1 Jam
# Pastikan kolom timestamp bertipe datetime
df_filtered['order_timestamp'] = pd.to_datetime(df_filtered['order_timestamp'], errors='coerce')

df_filtered = df_filtered.dropna(subset=['order_timestamp']).copy()



In [ ]:
df.info()

In [ ]:
prep = df_filtered['restaurant_preparation_time_minutes'].dropna()
print("Mean:", prep.mean())
print("Std Dev:", prep.std())
print("Median:", prep.median())
print("Skewness:", prep.skew())

In [ ]:
windowed_df = (
    df_filtered.groupby(
        ['restaurant_id', pd.Grouper(key='order_timestamp', freq='1H')]
    )
    .agg(
        # TARGET & ANTRIAN
        demand=('order_id', 'count'),
        service_process_mean=('restaurant_preparation_time_minutes', 'mean'),

        # FITUR ML (X) - Temporal
        order_hour=('order_hour', 'first'),
        day_of_week=('day_of_week', 'first'),
        is_weekend=('is_weekend', 'first'),

        # FITUR ML (X) - Kondisi Eksternal
        weather=('weather', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
        traffic_level=('traffic_level', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),

        # FITUR ML (X) - Karakteristik Pesanan
        total_items=('items_count', 'sum'),
        avg_subtotal=('subtotal', 'mean')
    )
    .reset_index()
)

# 3. Pembersihan nilai kosong akibat agregasi
windowed_df['service_process_mean'] = windowed_df.groupby('restaurant_id')['service_process_mean'].transform(lambda x: x.fillna(x.mean()))

In [ ]:
windowed_df.to_csv('cleaned_hourly_demand.csv', index=False)
windowed_df.head(10)

In [ ]:
windowed_df['order_hour'].value_counts().sort_index().plot(kind='bar', figsize=(10, 5), title='Distribusi Order per Jam')

In [ ]:
windowed_df['traffic_level'].value_counts().plot(kind='bar', figsize=(10, 5), title='Distribusi Level Lalu Lintas')

In [ ]:
# Mengatur gaya visualisasi
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))

# Membuat histogram dengan kurva KDE
sns.histplot(
    data=windowed_df, 
    x='service_process_mean', 
    kde=True, 
    color='royalblue', 
    bins=30
)

# Menambahkan judul dan label format yang rapi
plt.title('Sebaran Rata-Rata Waktu Layanan Per Jam (Service Time)', fontsize=14, fontweight='bold')
plt.xlabel('Rata-Rata Waktu Persiapan (Menit)', fontsize=12)
plt.ylabel('Frekuensi (Jumlah Jam-Restoran)', fontsize=12)

plt.show()

In [ ]:
windowed_df[windowed_df['demand'] < 3 ].head(10)